# COLAB 2: Processing multi-subject datasets with high-performance computing

## Course architecture

Data acquisition ➡️ Training an ML model ➡️ Visualizing results ➡️ Documenting research

## Instructions for the student
In this lab, you will train a machine learning (ML) model (convolutional neural network, CNN) on temporal EEG data recorded from multiple subjects and channels.

Your objective is to pass the explicit **"AI Prompt Challenges"** directly to an AI coding assistant, evaluate the structural properties of the returned functions, and paste them into the designated code boundaries.

## Module 0: Setting up an environment on FABRIC

**Student AI prompt challenge**

Copy and paste the code block from the cell below and the following text block into your AI assistant:

> "Give me commands I can use in the ubuntu terminal to (1) install anaconda, (2) create a conda environment, (3) activate that environment, (4) install numpy, pandas, matplotlib, pytorch (GPU-compatible)."

In [35]:
!nvidia-smi

Thu Aug 13 18:56:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             16W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [36]:
!pip install -q numpy pandas matplotlib

## Module 1: Data acquisition

We will continue working with the [EEG Emotion Recognition dataset](https://www.kaggle.com/datasets/wajahat1064/emotion-recognition-using-eeg-and-computer-games?resource=download), acquiring data from all 28 subjects. The cells below download the full dataset from Kaggle.

In [37]:
import kagglehub
import os

# Download from kaggle
path = kagglehub.dataset_download("https://www.kaggle.com/datasets/wajahat1064/emotion-recognition-using-eeg-and-computer-games")
print("Dataset downloaded to:", path)

path = os.path.join(path,
                     "Dataset - Emotion Recognition data Based on EEG Signals and Computer Games",
                     "Database for Emotion Recognition System Based on EEG Signals and Various Computer Games - GAMEEMO",
                     "GAMEEMO"
                    )

Using Colab cache for faster access to the 'emotion-recognition-using-eeg-and-computer-games' dataset.
Dataset downloaded to: /kaggle/input/emotion-recognition-using-eeg-and-computer-games


**Student AI prompt challenge**

The `subjects` variable below should hold subject identifiers from the dataset (S01-S28). Design a prompt and define this variable with the AI.

In [38]:
#TODO: your code here
subjects = [f"S{i:02d}" for i in range(1, 29)]

In [39]:
print(len(subjects))

28


The functions below read data from a single subjects and combine the resulting dataset into an array of size (subjects x game_types x channels x time).

In [40]:
import numpy as np
import pandas as pd
import os

def read_subject(subject, path):
    subject_recordings = []
    for game_idx in range(4):
        # Try both filename conventions: with and without 'Raw'
        csv_filename_raw = f'{subject}G{game_idx + 1}AllRawChannels.csv'
        csv_filename_standard = f'{subject}G{game_idx + 1}AllChannels.csv'

        file_path_raw = os.path.join(path, csv_filename_raw)
        file_path_standard = os.path.join(path, csv_filename_standard)

        if os.path.exists(file_path_raw):
            df = pd.read_csv(file_path_raw)
        elif os.path.exists(file_path_standard):
            df = pd.read_csv(file_path_standard)
        else:
            raise FileNotFoundError(f"Neither {file_path_raw} nor {file_path_standard} found.")

        df.drop("Unnamed: 14", axis = 1, inplace = True)

        # Shape into a channels x time array
        subject_recordings.append(df.to_numpy().T)

    # Shape into game_types x channels x time
    subject_recordings = np.stack(subject_recordings)

    return subject_recordings

def read_subjects(subjects, path):
    all_recordings = []
    for subject in subjects:
        subject_path = os.path.join(path, f"({subject})", "Raw EEG Data", ".csv format")
        data = read_subject(subject, subject_path)
        all_recordings.append(data)

    # Subjects x game_types x channels x time
    all_recordings = np.stack(all_recordings)
    return all_recordings

In [41]:
import os

base_path = "/root/.cache/kagglehub/datasets/wajahat1064/emotion-recognition-using-eeg-and-computer-games/versions/2"

for root, dirs, files in os.walk(base_path):
    if "S26" in root:
        print(root)
        for file in files[:10]:
            print("   ", file)

/root/.cache/kagglehub/datasets/wajahat1064/emotion-recognition-using-eeg-and-computer-games/versions/2/Dataset - Emotion Recognition data Based on EEG Signals and Computer Games/Database for Emotion Recognition System Based on EEG Signals and Various Computer Games - GAMEEMO/GAMEEMO/(S26)
/root/.cache/kagglehub/datasets/wajahat1064/emotion-recognition-using-eeg-and-computer-games/versions/2/Dataset - Emotion Recognition data Based on EEG Signals and Computer Games/Database for Emotion Recognition System Based on EEG Signals and Various Computer Games - GAMEEMO/GAMEEMO/(S26)/SAM Ratings
    G2.pdf
    G3.pdf
    G4.pdf
    G1.pdf
/root/.cache/kagglehub/datasets/wajahat1064/emotion-recognition-using-eeg-and-computer-games/versions/2/Dataset - Emotion Recognition data Based on EEG Signals and Computer Games/Database for Emotion Recognition System Based on EEG Signals and Various Computer Games - GAMEEMO/GAMEEMO/(S26)/Raw EEG Data
/root/.cache/kagglehub/datasets/wajahat1064/emotion-recogn

In [42]:
file_path = os.path.join(
    path,
    f"({subjects})",
    "Preprocessed EEG Data",
    ".csv format"
)

In [43]:
data = read_subjects(subjects, path)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/emotion-recognition-using-eeg-and-computer-games/Dataset - Emotion Recognition data Based on EEG Signals and Computer Games/Database for Emotion Recognition System Based on EEG Signals and Various Computer Games - GAMEEMO/GAMEEMO/(S01)/Raw EEG Data/.csv format/S01G1AllChannels.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Student AI prompt challenge**

As you can see, there is an error when reading data of one of the subjects. Design a prompt to list the contents of the subject's directory and compare the contents with antoher subject (e.g., subject S01). What is the problem?

In [ ]:
#TODO: your code here: list the contents of S26's directory and compare with S01


In [ ]:
import os

def list_subject_csv_directory(subject_id, base_path):
    subject_data_path = os.path.join(base_path, f"({subject_id})", "Raw EEG Data", ".csv format")
    print(f"Listing contents of: {subject_data_path}")
    if os.path.exists(subject_data_path):
        for item in os.listdir(subject_data_path):
            print(f"  - {item}")
    else:
        print(f"Directory does not exist: {subject_data_path}")

# Assuming 'path' variable from earlier cells is correctly defined and points to '.../GAMEEMO'
# For S01
list_subject_csv_directory('S01', path)

# For S26 (as a comparison, based on previous os.walk output)
list_subject_csv_directory('S26', path)

**Student AI prompt challenge**

Prompt AI to modify the list of subjects to exclude subject 26 (but still include subjects 1-25 and 27-28).

In [ ]:
#TODO: your code here


In [ ]:
data = read_subjects(subjects, path)

# Subjects x game_types x channels x time
print(data.shape)

## Module 2: Training an ML model

We are going to train a convolutional neural network (CNN) to predict the game type from the temporal data across channels of all available subjects.

**Student AI prompt challenge**

Copy and paste the code block from the cell below and the following text block into your AI assistant:

> "I am working with a dataset of size subjects x conditions x channels x time, my actual shape is (27, 4, 14, 38252). The data are currently stored as a numpy array. Write pytorch code that trains a simple CNN on that data to predict the condition (i.e., multiclass classification over 4 classes). Treat each subject x condition as a separate sample and make the CNN layers 1d (i.e., we convolve across time). The first layer should take 14 channels as evident from my data shape. Also leave 3 subjects out for validation (i.e., train only on 24 subjects) and add some one-time testing step (compute accuracy across the 3 subjects). Track loss and training accuracy so that I can plot them / save them later."

In [ ]:
#TODO: your code here


## Module 3: Visualizing results

**Student AI prompt challenge**

Design a prompt to plot the saved train and validation losses and accuracies.

In [ ]:
#TODO: your code here


## Module 4: Documenting research

**Student AI prompt challenge**
Design a prompt to save the resulting  train and validation losses and accuracies in a `.csv` file.

In [ ]:
#TODO: your code here


## Module 5: Additional prompt challenges

1. **Beat the Baseline:** Use AI to modify the CNN architecture (number of layers, kernel sizes, channels, pooling) and improve validation accuracy over the provided baseline.
2. **Try Different Optimizers:** Replace Adam with SGD, RMSprop, or AdamW. Compare convergence speed and final accuracy.
3. **Learning Rate Tuning:** Use AI to implement a learning rate scheduler (e.g., StepLR, CosineAnnealingLR, ReduceLROnPlateau) and determine whether it improves performance.
4. **Normalization Challenge:** Normalize the EEG channels (e.g., z-score using only the training subjects). Compare results with the unnormalized data.
5. **Confusion Matrix:** Compute and visualize the confusion matrix for the held-out subjects. Which conditions are most frequently confused?